In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from scipy.stats import randint

# add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features


In [2]:

# Load enriched dataset
df_feats, feature_cols = get_features("../data/raw")

# Filter seasons and minutes (igual que 03)
df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024) &
    (df_feats["minutes_played"] >= 100)
].copy()

df_ml.shape


(23442, 79)

In [3]:

df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) &
                 (df_ml["season_end_year"] <= 2022)].copy()

print("Train:", df_train.shape)
print("Val:", df_val.shape)

X_train = df_train[feature_cols]
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[feature_cols]
y_val = df_val["ballon_dor_winner"].astype(int)



Train: (14301, 79)
Val: (6038, 79)


In [4]:
print("Before SMOTE:", y_train.value_counts())

sm = SMOTE(k_neighbors=1, random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_res.value_counts())
X_train_res.shape


Before SMOTE: ballon_dor_winner
0    14290
1       11
Name: count, dtype: int64
After SMOTE: ballon_dor_winner
0    14290
1    14290
Name: count, dtype: int64


(28580, 72)

In [5]:
scaler = StandardScaler()

X_train_res_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)

In [12]:
param_dist = {
    "n_estimators": randint(300, 1500),
    "max_depth": [None] + list(range(5, 50, 5)),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],  # FIXED
    "bootstrap": [True, False]
}


In [13]:
rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=50,               # más = mejor, pero más lento
    scoring="roc_auc",       # métrica ideal para Balón d'Or
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

In [14]:
random_search.fit(X_train_res_scaled, y_train_res)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'bootstrap': [True, False], 'max_depth': [None, 5, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': <scipy.stats....0023D18738200>, ...}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [15]:
print("🔍 Best AUC (CV):", random_search.best_score_)
print("\n🏆 Best Hyperparameters:")
random_search.best_params_

🔍 Best AUC (CV): 0.9999999853067596

🏆 Best Hyperparameters:


{'bootstrap': False,
 'max_depth': 25,
 'max_features': 'log2',
 'min_samples_leaf': 1,
 'min_samples_split': 10,
 'n_estimators': 1447}

In [16]:
best_rf = random_search.best_estimator_

proba_val = best_rf.predict_proba(X_val_scaled)[:, 1]
pred_val = (proba_val >= 0.5).astype(int)

print("=== VALIDATION RESULTS (TUNED RF) ===")
print("AUC:", roc_auc_score(y_val, proba_val))
print("F1:", f1_score(y_val, pred_val))
print("Recall:", recall_score(y_val, pred_val))
print("Precision:", precision_score(y_val, pred_val))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, pred_val))

=== VALIDATION RESULTS (TUNED RF) ===
AUC: 0.9997238331952499
F1: 0.5
Recall: 0.3333333333333333
Precision: 1.0

Confusion Matrix:
[[6035    0]
 [   2    1]]


In [17]:
importances = best_rf.feature_importances_
idx = np.argsort(importances)[::-1]

importance_tuned = pd.DataFrame({
    "feature": [feature_cols[i] for i in idx],
    "importance": importances[idx]
})

importance_tuned.head(20)

,feature,importance
0,goals,0.089803
1,g_per90,0.083983
2,ga_per90,0.065533
3,goals_z,0.063021
4,g_per90_z,0.061114
5,ga_per90_z,0.060696
6,assists,0.052633
7,matches_played,0.044703
8,a_per90,0.043691
9,a_per90_z,0.043048


In [18]:

from sklearn.metrics import precision_recall_curve

prec, rec, th = precision_recall_curve(y_val, proba_val)

thr_df = pd.DataFrame({
    "threshold": th,
    "precision": prec[:-1],
    "recall": rec[:-1],
    "f1": 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1])
})

thr_df.sort_values("f1", ascending=False).head(10)


,threshold,precision,recall,f1
362,0.162539,0.666667,0.666667,0.666667
358,0.051942,0.428571,1.000000,0.600000
361,0.142328,0.500000,0.666667,0.571429
357,0.049242,0.375000,1.000000,0.545455
360,0.130506,0.400000,0.666667,0.500000
356,0.037633,0.333333,1.000000,0.500000
364,0.654563,1.000000,0.333333,0.500000
355,0.036097,0.300000,1.000000,0.461538
359,0.110447,0.333333,0.666667,0.444444
354,0.033770,0.272727,1.000000,0.428571


In [20]:
best_threshold = 0.162539

pred_val_adj = (proba_val >= best_threshold).astype(int)

print("=== USING OPTIMAL THRESHOLD ===")
print("Threshold:", best_threshold)
print("AUC:", roc_auc_score(y_val, proba_val))
print("Recall:", recall_score(y_val, pred_val_adj))
print("Precision:", precision_score(y_val, pred_val_adj))
print("F1:", f1_score(y_val, pred_val_adj))
print(confusion_matrix(y_val, pred_val_adj))


=== USING OPTIMAL THRESHOLD ===
Threshold: 0.162539
AUC: 0.9997238331952499
Recall: 0.6666666666666666
Precision: 0.6666666666666666
F1: 0.6666666666666666
[[6034    1]
 [   1    2]]


In [21]:
from joblib import dump

best_threshold = 0.162539

dump(
    {
        "model": best_rf,
        "scaler": scaler,
        "feature_cols": feature_cols,
        "threshold": best_threshold
    },
    "rf_best_tuned.pkl"
)


['rf_best_tuned.pkl']